https://www.bilibili.com/video/BV1MGrTYVEXq/

In [1]:
import torch 
import torch.nn as nn
import math 

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, d_model, n_head, n_layer, d_ff, dropout=0.1, max_length=50_000):
        super().__init__()
        self.d_model = d_model
        self.vocab_emb = nn.Embedding(vocab_size, d_model)
        # a simple linear pos encoding
        # self.pos_encoding = nn.Embedding(max_length, d_model)
        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=n_head,
            num_encoder_layers=n_layer,
            num_decoder_layers=n_layer,
            dim_feedforward=d_ff,
            dropout=dropout
        )
        self.ow = nn.Linear(d_model, vocab_size)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('pos_encoding', self.create_sin_cos_position_embedding(max_length, d_model))
        
    def create_sin_cos_position_embedding(self, max_length, d_model):
        position = torch.arange(0, max_length).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10_000.0) / d_model))
        pe = torch.zeros(max_length, d_model)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(1)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        src_seq_len = src.size(-2)
        tgt_seq_len = tgt.size(-2)
        src = self.dropout(self.vocab_emb(src) + self.pos_encoding[:src_seq_len, :])
        tgt = self.dropout(self.vocab_emb(tgt) + self.pos_encoding[:tgt_seq_len, :])
        out = self.transformer(src, tgt, src_mask=src_mask, tgt_mask=tgt_mask)
        return self.ow(out)


In [2]:

# Example usage
vocab_size = 10000
embed_size = 512
num_heads = 8
num_encoder_layers = 6
num_decoder_layers = 6
forward_expansion = 2048
dropout = 0.1
max_length = 100
    
model = TransformerModel(
    vocab_size,
    embed_size,
    num_heads,
    num_encoder_layers,
    forward_expansion,
    dropout,
    max_length
)

# Dummy input
src = torch.randint(0, vocab_size, (50, 32))  # (source sequence length, batch size)
trg = torch.randint(0, vocab_size, (50, 32))  # (target sequence length, batch size)

output = model(src, trg)
print(output.shape)  # Expected shape: (target sequence length, batch size, vocab size)


/home/kennethwang/miniconda3/envs/py-notebook/lib/python3.12/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


torch.Size([50, 32, 10000])


In [ ]:
# Load TED talks dataset for Portuguese-English translation
import tensorflow_datasets as tfds
import tensorflow as tf
import torch.optim as optim
import time

# Load the dataset
train_examples, val_examples, test_examples = tfds.load(
    'ted_hrlr_translate/pt_to_en',
    split=['train', 'validation', 'test'],
    as_supervised=True)

# Create tokenizers
tokenizers = {}
for lang, data in [('pt', [ex[0].numpy().decode('utf-8') for ex in train_examples]), 
                   ('en', [ex[1].numpy().decode('utf-8') for ex in train_examples])]:
    tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
        data, target_vocab_size=5000)  # Reduced vocab size
    tokenizers[lang] = tokenizer

# Constants for special tokens
START_TOKEN = [tokenizers['pt'].vocab_size]
END_TOKEN = [tokenizers['pt'].vocab_size + 1]
VOCAB_SIZE_PT = tokenizers['pt'].vocab_size + 2
VOCAB_SIZE_EN = tokenizers['en'].vocab_size + 2

# Preprocessing function
def preprocess_text(pt, en):
    pt = START_TOKEN + tokenizers['pt'].encode(pt.numpy().decode('utf-8')) + END_TOKEN
    en = START_TOKEN + tokenizers['en'].encode(en.numpy().decode('utf-8')) + END_TOKEN
    return pt, en

def tf_preprocess(pt, en):
    return tf.py_function(preprocess_text, [pt, en], [tf.int64, tf.int64])

# Create TF datasets with reduced sizes
BUFFER_SIZE = 1000  # Reduced buffer size
BATCH_SIZE = 4  # Reduced batch size
MAX_LENGTH = 20  # Reduced max sequence length

def filter_max_length(pt, en):
    return tf.logical_and(tf.size(pt) <= MAX_LENGTH,
                         tf.size(en) <= MAX_LENGTH)

train_dataset = (train_examples
                .map(tf_preprocess)
                .filter(filter_max_length)
                .shuffle(BUFFER_SIZE)
                .padded_batch(BATCH_SIZE, padded_shapes=([None], [None]))
                .prefetch(tf.data.AUTOTUNE))

val_dataset = (val_examples
              .map(tf_preprocess)
              .filter(filter_max_length)
              .padded_batch(BATCH_SIZE, padded_shapes=([None], [None])))

# Convert TF dataset to PyTorch
def tf_to_torch(tf_dataset):
    for pt_batch, en_batch in tf_dataset:
        pt_tensor = torch.LongTensor(pt_batch.numpy())
        en_tensor = torch.LongTensor(en_batch.numpy())
        yield pt_tensor.T, en_tensor.T  # Transpose to match PyTorch expected shape

# Training function
def train_epoch(model, optimizer, criterion, train_data, device):
    model.train()
    losses = 0.
    for src, tgt in tf_to_torch(train_data):
        src = src.to(device)
        tgt = tgt.to(device)
        
        # Create masks
        src_mask = None  # For encoder self-attention
        tgt_len = tgt.shape[0] - 1  # Adjust for teacher forcing
        # Create square subsequent mask for decoder self-attention
        tgt_mask = torch.triu(torch.ones(tgt_len, tgt_len) * float('-inf'), diagonal=1).to(device)
        
        optimizer.zero_grad()
        output = model(src, tgt[:-1], src_mask, tgt_mask)
        
        # Ensure target values are within valid range
        tgt_labels = tgt[1:].reshape(-1)
        tgt_labels = torch.clamp(tgt_labels, 0, VOCAB_SIZE_EN - 1)
        
        loss = criterion(output.reshape(-1, output.shape[-1]), tgt_labels)
        loss.backward()
        optimizer.step()
        
        losses += loss.item()
        # print(f"loss{loss.item()}")
    
    return losses / len(list(train_data))

# Training setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# Update model parameters for new vocabulary sizes
model.vocab_emb = nn.Embedding(VOCAB_SIZE_PT, embed_size)
model.ow = nn.Linear(embed_size, VOCAB_SIZE_EN)
model = model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.0001, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Using 0 as padding index

# Training loop
NUM_EPOCHS = 1
for epoch in range(NUM_EPOCHS):
    start_time = time.time()
    train_loss = train_epoch(model, optimizer, criterion, train_dataset, device)
    end_time = time.time()
    
    print(f"Epoch: {epoch+1}, Train loss: {train_loss:.3f}, Epoch time: {(end_time - start_time):.2f}s")


2025-03-30 20:17:23.491204: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743391043.502520    2885 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743391043.505748    2885 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-30 20:17:23.516954: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1743391045.904442    2885 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 90

In [8]:
train_dataset.size()

AttributeError: '_PrefetchDataset' object has no attribute 'size'

In [6]:
def evaluate(model, val_data, device):
    model.eval()
    predictions = []
    targets = []
    
    with torch.no_grad():
        for i, (src, tgt) in enumerate(val_data):
            if i >= 10:  # Only evaluate first 10 samples
                break
                
            src = src.to(device)
            tgt = tgt.to(device)
            
            # Get model prediction
            output = model(src, tgt[:-1], None, None)
            pred = output.argmax(dim=-1)
            
            predictions.append(pred.cpu().numpy())
            targets.append(tgt[1:].cpu().numpy())
    
    print("\nValidation Results:")
    print("-" * 50)
    for pred, target in zip(predictions, targets):
        pred_text = " ".join([idx_to_word_en[idx] for idx in pred.flatten() if idx in idx_to_word_en])
        target_text = " ".join([idx_to_word_en[idx] for idx in target.flatten() if idx in idx_to_word_en])
        print(f"\nPredicted: {pred_text}")
        print(f"Target   : {target_text}")
        print("-" * 50)

# Evaluate model on validation data
print("Evaluating model on 10 validation samples...")
evaluate(model, val_dataset, device)


Evaluating model on 10 validation samples...


2025-03-30 20:05:56.879100: W tensorflow/core/kernels/data/cache_dataset_ops.cc:914] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset  will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`. You should use `dataset.take(k).cache().repeat()` instead.


AttributeError: 'tensorflow.python.framework.ops.EagerTensor' object has no attribute 'to'

In [4]:
import torch

# Create a sample tensor
x = torch.tensor([[1, 2, 3],
                  [4, 5, 6],
                  [7, 8, 9]])

print("Original tensor:")
print(x)
print(f"Shape: {x.shape}\n")

# 1. Transpose - Swap dimensions
x_t = x.transpose(0, 1)  # Swap rows and columns
print("x.transpose(0, 1)")
print("After transpose:")
print(x_t)
print(f"Shape: {x_t.shape}\n")

# 2. Reshape - Change dimensions while maintaining total elements
x_r = x.reshape(1, 9)  # Reshape to 1x9
print("x.reshape(1, 9)")
print("After reshape to 1x9:")
print(x_r)
print(f"Shape: {x_r.shape}\n")

# 3. Squeeze - Remove dimensions of size 1
x_s = x_r.squeeze(0)  # Remove first dimension
print("x_r.squeeze(0)")
print("After squeeze:")
print(x_s)
print(f"Shape: {x_s.shape}\n")

# 4. Unsqueeze - Add a dimension of size 1
x_u = x_s.unsqueeze(0)  # Add dimension at index 0
print("x_s.unsqueeze(0)")
print("After unsqueeze:")
print(x_u)
print(f"Shape: {x_u.shape}\n")

# Common use cases
# 1. Adding batch dimension
batch = torch.unsqueeze(x, 0)  # Add batch dimension
print("torch.unsqueeze(x, 0)")
print("Adding batch dimension:")
print(batch)
print(f"Shape: {batch.shape}\n")

# 2. Reshaping for attention
# From [batch, seq_len, hidden_dim] to [batch, seq_len, n_heads, head_dim]
batch_size, seq_len, hidden_dim = 2, 3, 4
x_attn = torch.randn(batch_size, seq_len, hidden_dim)
x_attn_reshaped = x_attn.reshape(batch_size, seq_len, 2, 2)  # Assuming 2 attention heads
print("x_attn.reshape(batch_size, seq_len, 2, 2)")
print("Reshaping for attention:")
print(f"Original shape: {x_attn.shape}")
print(f"Reshaped: {x_attn_reshaped.shape}")


Original tensor:
tensor([[1, 2, 3],
        [4, 5, 6],
        [7, 8, 9]])
Shape: torch.Size([3, 3])

x.transpose(0, 1)
After transpose:
tensor([[1, 4, 7],
        [2, 5, 8],
        [3, 6, 9]])
Shape: torch.Size([3, 3])

x.reshape(1, 9)
After reshape to 1x9:
tensor([[1, 2, 3, 4, 5, 6, 7, 8, 9]])
Shape: torch.Size([1, 9])

x_r.squeeze(0)
After squeeze:
tensor([1, 2, 3, 4, 5, 6, 7, 8, 9])
Shape: torch.Size([9])

x_s.unsqueeze(0)
After unsqueeze:
tensor([[1, 2, 3, 4, 5, 6, 7, 8, 9]])
Shape: torch.Size([1, 9])

torch.unsqueeze(x, 0)
Adding batch dimension:
tensor([[[1, 2, 3],
         [4, 5, 6],
         [7, 8, 9]]])
Shape: torch.Size([1, 3, 3])

x_attn.reshape(batch_size, seq_len, 2, 2)
Reshaping for attention:
Original shape: torch.Size([2, 3, 4])
Reshaped: torch.Size([2, 3, 2, 2])


In [3]:
torch.triu(torch.ones(20, 20), diagonal=1).bool().unsqueeze(0).unsqueeze(0).size()

torch.Size([1, 1, 20, 20])